In [43]:
import configparser
from os import getcwd
import pandas as pd
from yfpy.query import YahooFantasySportsQuery
from sqlalchemy.dialects.postgresql.base import PGDialect; PGDialect._get_server_version_info = lambda *args: (9, 2)
from dataHub import dataHub
dh = dataHub()

db_con = dh.db_connect('postgre')

parser = configparser.ConfigParser()
parser.read(getcwd() + '/database.ini')
yahoo_creds = dict(parser.items('yahoo'))
yahoo_creds['token_time'] = float(yahoo_creds['token_time'])

query = YahooFantasySportsQuery(
    league_id=yahoo_creds['league_id'],
    game_code="nba",
    yahoo_access_token_json=yahoo_creds
)

In [65]:
df = pd.read_sql('SELECT DISTINCT season, platform, league_id, league_name FROM fty.league_competitor', db_con)
# df.to_sql('league', db_con, schema='fty', index=False, if_exists='replace')

3

In [24]:
# league_info

# season, league_id, league_name,         competitor_id,  competitor_abbrev, competitor_name
# 2023-24 1966813226 Let's Get Tropical	  1	              5 ⭐ 	             Philly 5 ⭐️ Men
# 2023-24 1966813226 Let's Get Tropical	  2	              Poop	             El Barto 	

dfs = []

# Replace with season fron nba_api
season = query.get_current_game_metadata().season
season = str(season) + '-' + str(season+1-2000)


competitor_id = query.get_league_teams()[0].team_id
competitor_abbrev = query.get_league_teams()[0].managers[0].nickname
competitor_name = query.get_league_teams()[0].name.decode()

for competitor in query.get_league_teams():

    dfs.append(pd.DataFrame({
        'season': [season],
        'season_year': [2024],
        'platform': 'Yahoo',
        'league_id': [yahoo_creds['league_id']],
        'league_name': [query.get_league_metadata().name.decode()],
        'competitor_id': [competitor.team_id],
        'competitor_abbrev': [competitor.managers[0].nickname],
        'competitor_name': [competitor.name.decode()],
        'division_id': [None],
        'division_name': [None]
    }))

df = pd.concat(dfs)
df

2024-10-12 18:09:05.043 - WARNING - query.py - yfpy.query:1030 - No game id or season/year provided, defaulting to current fantasy season.
2024-10-12 18:09:06.520 - WARNING - query.py - yfpy.query:1030 - No game id or season/year provided, defaulting to current fantasy season.
2024-10-12 18:09:08.058 - WARNING - query.py - yfpy.query:1030 - No game id or season/year provided, defaulting to current fantasy season.
2024-10-12 18:09:09.588 - WARNING - query.py - yfpy.query:1030 - No game id or season/year provided, defaulting to current fantasy season.
2024-10-12 18:09:11.229 - WARNING - query.py - yfpy.query:1030 - No game id or season/year provided, defaulting to current fantasy season.
2024-10-12 18:09:12.692 - WARNING - query.py - yfpy.query:1030 - No game id or season/year provided, defaulting to current fantasy season.
2024-10-12 18:09:14.124 - WARNING - query.py - yfpy.query:1030 - No game id or season/year provided, defaulting to current fantasy season.
2024-10-12 18:09:15.534 - W

,season,season_year,platform,league_id,league_name,competitor_id,competitor_abbrev,competitor_name,division_id,division_name
0,2024-25,2024,Yahoo,121793,Let's Get Tropical,1,Josh,Philly 5 ⭐️ Men,None,None
0,2024-25,2024,Yahoo,121793,Let's Get Tropical,2,Oli,shaggy_camel,None,None
0,2024-25,2024,Yahoo,121793,Let's Get Tropical,3,Cameron,Cameron's Choice Team,None,None
0,2024-25,2024,Yahoo,121793,Let's Get Tropical,4,Step2Cool,draftbyslo,None,None
0,2024-25,2024,Yahoo,121793,Let's Get Tropical,5,Jeremy,Palmy Slizzards,None,None
0,2024-25,2024,Yahoo,121793,Let's Get Tropical,6,Jeremy,Jeremy's Incredible Team,None,None
0,2024-25,2024,Yahoo,121793,Let's Get Tropical,7,jason,jason's Spectacular Team,None,None
0,2024-25,2024,Yahoo,121793,Let's Get Tropical,8,Jess,Taranaki Thunder,None,None


In [25]:
# df.to_sql('league_info', db_con, schema='fty', index=False, if_exists='append')

8

In [92]:
# league_schedule
for week in range(1, query.get_league_metadata().end_week+1):
    query.get_league_matchups_by_week(week)

In [93]:
# matchup_box_score
query.get_team_stats_by_week()

TypeError: YahooFantasySportsQuery.get_team_stats_by_week() missing 1 required positional argument: 'team_id'

In [80]:
# Injury and Free agent status of players, competitor roster

query.get_league_players()[0].ownership
query.get_league_players()[0].status
query.get_league_players()[0].status_full

2024-10-04 22:40:38.334 - ERROR - query.py - yfpy.query:543 - No data found when attempting extraction from fields: ['league', 'players']


Ownership({})